In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import scipy.stats as stats
from scipy.stats import levene
import statsmodels.stats.multitest as smm

import sys
import sys
sys.path.append('/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/clinical_data_analysis/statistical_analysis')
from utilis import *
sys.path.append('../')
from settings import *

In [ ]:
DATA_PATH = '/ems/elsc-labs/habib-n/yuval.rom/BCG/clinical_data/Data/fightAD_general_data_table.xlsx'
SAVE_DIR = 'figures/'
df = pd.read_excel(DATA_PATH, sheet_name='1')
df.columns = df.loc[0]
df = df.drop(0)
df = df.iloc[:,:-2]
print(df.shape)
df.head(1)

In [ ]:
df.rename(columns={'#':'ID',
                   'תאריך בדיקות דם': 'blood_test_date',
                   'שעת לקיחת דם':'blood_taken_time',
                   'מספר מנות':'num_of_injections',
                   'גיל הפסקת מחזור':'menopause',
                   'אסטרוגן':'estorgen',
                   'Frontal memory score (#1)- כמה מילים זכר/ה ברמז הראשון. נקודה אם זכר/ה רמז ראשון ושתי נקודות אם לא.':'Frontal memory score (#1)',
                   'Hipocampal memory score (#2)- כמה מילים לא זכר/ה בכלל':'Hipocampal memory score (#2)',
                   'p-tau_ new_#1':'p-tau_new_#1',
                   'ביקורת אחרי שנה':'checkup after a year',
                   'חיסון':'vaccine',
                   'תרופות':'medications',
                   'הערות':'notes',
                   'Number_of_risk_faktors':'Number_of_risk_factors',
                   'P-tau217':'p-tau217',
                   'צרבונין':'Cerebonin',
                   'Spatial ': 'Spatial'
                   }, inplace=True)

In [ ]:
df = df.apply(pd.to_numeric, errors='ignore')

# Outliers

In [ ]:
df.iloc[df[TAU_217].argmax()]
outliers = []  # Participant exclusions are maintained privately.

print(df.shape) 
df = df.drop(df[df['ID'].isin(outliers)].index)
print(df.shape) 

In [ ]:
(df['p-tau217'] > HIGH_TAU_217_THR).value_counts()

# Add columns

In [ ]:
df = df[df[MOCA_COL] != '?']
df['MCI'] = df[MOCA_COL] < MCI_THR
df[TAU_CAT] = pd.cut(df[TAU_217], bins=[0, INT_TAU_217_THR, HIGH_TAU_217_THR, np.inf], labels=['Low', 'Intermediate', 'High'])
df_full = df[df[MOCA_COL].notna()].copy()

df = df[df[TAU_217].notna()] # Filter only to those with tau_217 values
# removing samples with moca below the minimum threshold of 19
df = df[df[MOCA_COL] > MIN_MOCA_SCORE]
df_high_vs_low = df[(df[TAU_217] > HIGH_TAU_217_THR) | (df[TAU_217] < INT_TAU_217_THR)]
df_high_vs_low[TAU_CAT] = df_high_vs_low[TAU_CAT].cat.remove_categories(['Intermediate'])

In [ ]:
df_high_vs_low[TAU_CAT].value_counts()

## Risk Factors

In [ ]:
RISK_FACTOR_DIR = SAVE_DIR + 'risk_factors/'

### Total number of risk factors (not increased in pre-clinical AD as compared to low p-tau matched controls).

In [ ]:
N_RISK_COL = 'Number_of_risk_factors'
df_high_vs_low_risk = df_high_vs_low.dropna(subset=[N_RISK_COL])
df_high_vs_low_risk[N_RISK_COL] = df_high_vs_low_risk[N_RISK_COL].astype(int)
grpah_colors = [healthy_color, risk_color]

In [ ]:
# stacked histogram of number of risk factors
sns.histplot(data=df_high_vs_low_risk, x=N_RISK_COL, hue=TAU_CAT, multiple='stack', bins=9, palette=grpah_colors, kde=True)
plt.legend(labels=['High Tau', 'Low Tau'])

plt.title('Number of risk factors vs Tau 217 category')
plt.savefig(RISK_FACTOR_DIR + 'Number_of_risk_factors_distribution.pdf')


In [ ]:
kdeplot_high_vs_low(df_high_vs_low_risk, N_RISK_COL, TAU_CAT, graph_colors, 'Number of risk factors vs Tau 217 category',RISK_FACTOR_DIR + 'Number_of_risk_factors_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_risk[df_high_vs_low_risk[TAU_CAT] == 'Low'][N_RISK_COL]
group2 = df_high_vs_low_risk[df_high_vs_low_risk[TAU_CAT] == 'High'][N_RISK_COL]

p_number_of_risk_factors = stat_test(data_type, group1, group2)

### Anexity and Depression

In [ ]:
df_high_vs_low_anx = df_high_vs_low.dropna(subset=[ANEXITY_COL])
grpah_colors = [healthy_color, risk_color] * 2

plot_histplot(df_high_vs_low_anx, ANEXITY_COL, TAU_CAT, 2 , graph_colors, 'Anxiety/Depression vs Tau 217 category', RISK_FACTOR_DIR + 'Anxiety_Depression_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_anx
group2 = [TAU_CAT, ANEXITY_COL]

p_anxiety = stat_test(data_type, group1, group2) # Too few samples for the test

### BMI

In [ ]:
BMI = 'BMI'

df_high_vs_low_bmi = df_high_vs_low.dropna(subset=[BMI])
df_high_vs_low_bmi = df[pd.to_numeric(df[BMI], errors='coerce').notna()]
df_high_vs_low_bmi[BMI] = df_high_vs_low_bmi[BMI].astype(float)
grpah_colors = [healthy_color, risk_color]

In [ ]:
b = ((df_high_vs_low[TAU_CAT] == 'Low')).sum()
a = ((df_high_vs_low[TAU_CAT] == 'High')).sum()
a, b

In [ ]:
mask = df['BMI'].apply(lambda x: isinstance(x, pd.Timestamp))
mask.sum()

# df_high_vs_low['BMI'].is()

In [ ]:
kdeplot_high_vs_low(df_high_vs_low_bmi, BMI, TAU_CAT, graph_colors, 'BMI vs Tau 217 category', RISK_FACTOR_DIR + 'BMI_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_bmi[df_high_vs_low_bmi[TAU_CAT] == 'Low'][BMI]
group2 = df_high_vs_low_bmi[df_high_vs_low_bmi[TAU_CAT] == 'High'][BMI]

p_bmi = stat_test(data_type, group1, group2)

###   Heart disease

In [ ]:
df['Heart_disease'].value_counts()

In [ ]:
HD = 'Heart_disease'

df_high_vs_low_hd = df_high_vs_low.dropna(subset=[HD])
grpah_colors = [healthy_color, risk_color]

In [ ]:
plot_histplot(df_high_vs_low_hd, HD, TAU_CAT, 2, graph_colors, 'Heart disease vs Tau 217 category', RISK_FACTOR_DIR + 'Heart_disease_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_hd
group2 = [TAU_CAT, HD]

p_heart_disease = stat_test(data_type, group1, group2) # Too few samples for the test

### Diabetes

In [ ]:
DIABETES = 'Diabetes'
df_high_vs_low_d = df_high_vs_low.dropna(subset=[DIABETES])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_d, DIABETES, TAU_CAT, 2 , graph_colors, 'Diabetes vs Tau 217 category', RISK_FACTOR_DIR + 'Diabetes_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_d
group2 = [TAU_CAT, DIABETES]

p_diabetes = stat_test(data_type, group1, group2) # Too few samples for the test

###  Hypertension    	

In [ ]:
HYP = 'Hypertension'

df_high_vs_low_hyp = df_high_vs_low.dropna(subset=[HYP])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_d, HYP, TAU_CAT, 2, grpah_colors, 'Hypertension vs Tau 217 category', RISK_FACTOR_DIR + 'Hypertension_vs_Tau_217_category_violin.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_hyp
group2 = [TAU_CAT, HYP]

p_hypertenstion = stat_test(data_type, group1, group2) # Too few samples for the test

### Hyperlipidemia

In [ ]:
HL = 'Hyperlipidemia'

df_high_vs_low_hl = df_high_vs_low.dropna(subset=[HL])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_hl, HL, TAU_CAT, 2, grpah_colors, 'Hyperlipidemia vs Tau 217 category', RISK_FACTOR_DIR + 'Hyperlipidemia_vs_Tau_217_category_hist.pdf') 

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_hl
group2 = [TAU_CAT, HL]

p_hyperlipidemia = stat_test(data_type, group1, group2)

### Periodontal disease

In [ ]:
PERI = 'Periodontal disease'

df_high_vs_low_peri = df_high_vs_low.dropna(subset=[PERI])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_peri, PERI, TAU_CAT, 2, grpah_colors, 'Periodontal disease vs Tau 217 category', RISK_FACTOR_DIR + 'Periodontal_disease_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_peri
group2 = [TAU_CAT, PERI]

p_peridontal = stat_test(data_type, group1, group2) # Too few samples for the test

### Recurrent_UTI/_pneumonia/other_infection

In [ ]:
UTI = 'Recurrent_UTI/_pneumonia/other_infection'

df_high_vs_low_uti = df_high_vs_low.dropna(subset=[UTI])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_uti, UTI, TAU_CAT, 2, grpah_colors, 'Recurrent UTI/pneumonia/other infection vs Tau 217 category', RISK_FACTOR_DIR + 'Recurrent_UTI_pneumonia_other_infection_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_uti
group2 = [TAU_CAT, UTI]

p_uti = stat_test(data_type, group1, group2) # Too few samples for the test

### Systemic_inflammatory_disease

In [ ]:
SID = 'Systemic_inflammatory_disease'

df_high_vs_low_sid = df_high_vs_low.dropna(subset=[SID])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_sid, SID, TAU_CAT, 2, grpah_colors, 'Systemic inflammatory disease vs Tau 217 category', RISK_FACTOR_DIR + 'Systemic_inflammatory_disease_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_sid
group2 = [TAU_CAT, SID]

p_systemic_inflammatory = stat_test(data_type, group1, group2) # Too few samples for the test

### Smoking 


In [ ]:
SMOKING = 'Smoking'

df_high_vs_low_smoking = df_high_vs_low.dropna(subset=[SMOKING])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_smoking, SMOKING, TAU_CAT, 2, grpah_colors, 'Smoking vs Tau 217 category', RISK_FACTOR_DIR + 'Smoking_vs_Tau_217_category_hist.pdf')

In [ ]:
df_high_vs_low['Smoking'].unique()

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_smoking[df_high_vs_low_smoking[TAU_CAT] == 'Low'][SMOKING].replace({'NO': 0, 'in_the_past': 1, 'in _the_ past': 1, 'YES': 1})
group2 = df_high_vs_low_smoking[df_high_vs_low_smoking[TAU_CAT] == 'High'][SMOKING].replace({'NO': 0, 'in_the_past': 1, 'in _the_ past': 1, 'YES': 1})

p_smoking = stat_test(data_type, group1, group2)

### Pack_years 

In [ ]:
PACK_YEARS = 'Pack_years'

df_high_vs_low_pack_years = df_high_vs_low.dropna(subset=[PACK_YEARS])
df_high_vs_low_pack_years = df_high_vs_low_pack_years[df_high_vs_low_pack_years[PACK_YEARS] > 0]
grpah_colors = [healthy_color, risk_color]

kdeplot_high_vs_low(df_high_vs_low_pack_years, PACK_YEARS, TAU_CAT, graph_colors, 'Pack years vs Tau 217 category', RISK_FACTOR_DIR + 'Pack_years_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_pack_years[df_high_vs_low_pack_years[TAU_CAT] == 'Low'][PACK_YEARS]
group2 = df_high_vs_low_pack_years[df_high_vs_low_pack_years[TAU_CAT] == 'High'][PACK_YEARS]

p_pack_years = stat_test(data_type, group1, group2)

### Alcohol_consumption

In [ ]:
ALCOHOL = 'Alcohol_consumption'

df_high_vs_low_alcohol = df_high_vs_low.dropna(subset=[ALCOHOL])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_alcohol, ALCOHOL, TAU_CAT, 2, grpah_colors, 'Alcohol consumption vs Tau 217 category', RISK_FACTOR_DIR + 'Alcohol_consumption_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_alcohol
group2 = [TAU_CAT, ALCOHOL]

p_alcohol = stat_test(data_type, group1, group2) # Too few samples for the test

###  Insomnia

In [ ]:
INSOMNIA = 'Insomnia'

df_high_vs_low_insomnia = df_high_vs_low.dropna(subset=[INSOMNIA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_insomnia, INSOMNIA, TAU_CAT, 2, grpah_colors, 'Insomnia vs Tau 217 category', RISK_FACTOR_DIR + 'Insomnia_vs_Tau_217_category_hist.pdf')

In [ ]:
# STATISTICAL TEST
data_type = 'Categorical'

group1 = df_high_vs_low_insomnia[df_high_vs_low_insomnia[INSOMNIA] != 'in_the_past'] # Only one subject has 'in_the_past' value
group2 = [TAU_CAT, INSOMNIA]

p_insomnia = stat_test(data_type, group1, group2)

### OSA 

In [ ]:
OSA = 'OSA'

df_high_vs_low_osa = df_high_vs_low.dropna(subset=[OSA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_osa, OSA, TAU_CAT, 2, grpah_colors, 'OSA vs Tau 217 category', RISK_FACTOR_DIR + 'OSA_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_osa
group2 = [TAU_CAT, OSA]

p_osa = stat_test(data_type, group1, group2)

### severity of OSA

In [ ]:
OSA_SEVERITY = 'severity of OSA'

df_high_vs_low_osa_severity = df_high_vs_low.dropna(subset=[OSA_SEVERITY])
df_high_vs_low_osa_severity = df_high_vs_low_osa_severity[df_high_vs_low_osa_severity[OSA_SEVERITY] > 0]
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_osa_severity, OSA_SEVERITY, TAU_CAT, 3, grpah_colors, 'OSA severity vs Tau 217 category', RISK_FACTOR_DIR + 'OSA_severity_vs_Tau_217_category_hist.pdf')

In [ ]:
#Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_osa_severity[df_high_vs_low_osa_severity[TAU_CAT] == 'Low'][OSA_SEVERITY]
group2 = df_high_vs_low_osa_severity[df_high_vs_low_osa_severity[TAU_CAT] == 'High'][OSA_SEVERITY]

p_severity_osa = stat_test(data_type, group1, group2)

### Physical activity

In [ ]:
PA = 'Physical activity'

df_high_vs_low[PA].replace({'LOW ': 'LOW', 'HIGH ': 'HIGH', 'MEDIUM ': 'MEDIUM'}, inplace=True)

In [ ]:
PA = 'Physical activity'

df_high_vs_low_pa = df_high_vs_low.dropna(subset=[PA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_pa, PA, TAU_CAT, 2, grpah_colors, 'Physical activity vs Tau 217 category', RISK_FACTOR_DIR + 'Physical_activity_vs_Tau_217_category_hist.pdf')

In [ ]:
sns.histplot(df_high_vs_low_pa, x=PA, hue=TAU_CAT, multiple='stack', bins=3, palette=grpah_colors, kde=True, fill=True)
plt.title('Physical activity vs Tau 217 category')

In [ ]:
# Statistical test
data_type = 'Ordinal'

group1 = df_high_vs_low_pa[df_high_vs_low_pa[TAU_CAT] == 'Low'][PA].replace({'NO': 0, 'LOW': 1, 'MEDIUM': 2, 'HIGH': 3})
group2 = df_high_vs_low_pa[df_high_vs_low_pa[TAU_CAT] == 'High'][PA].replace({'NO': 0, 'LOW': 1, 'MEDIUM': 2, 'HIGH': 3})

p_physical_activity = stat_test(data_type, group1, group2)

### Time_Physical_activity

In [ ]:
TIME_PA = 'Time_Physical_activity_(hours/week)'

df_high_vs_low_time_pa = df_high_vs_low.dropna(subset=[TIME_PA])
grpah_colors = [healthy_color, risk_color]
kdeplot_high_vs_low(df_high_vs_low_time_pa, TIME_PA, TAU_CAT, graph_colors, 'Time Physical activity vs Tau 217 category', RISK_FACTOR_DIR + 'Time_Physical_activity_vs_Tau_217_category_kde.pdf', alpha=0.5)

In [ ]:
# Statistical test
data_type = 'Continuous'

group1 = df_high_vs_low_time_pa[df_high_vs_low_time_pa[TAU_CAT] == 'Low'][TIME_PA]
group2 = df_high_vs_low_time_pa[df_high_vs_low_time_pa[TAU_CAT] == 'High'][TIME_PA]

p_time_physical_activity = stat_test(data_type, group1, group2)

### Cognitive_activity 

In [ ]:
CA = 'Cognitive_activity'

df_high_vs_low_ca = df_high_vs_low.dropna(subset=[CA])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ca, CA, TAU_CAT, 2, grpah_colors, 'Cognitive activity vs Tau 217 category', RISK_FACTOR_DIR + 'Cognitive_activity_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test
data_type = 'Categorical'

group1 = df_high_vs_low_ca
group2 = [TAU_CAT, CA]

p_cognitive_activity = stat_test(data_type, group1, group2) # Too few samples for the test

### AD_in_the_family

In [ ]:
AD_IF = 'AD_in_the_family'

df_high_vs_low_ad_ip = df_high_vs_low.dropna(subset=[AD_IF])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_ad_ip, AD_IF, TAU_CAT, 2, grpah_colors, 'AD in the family vs Tau 217 category', RISK_FACTOR_DIR + 'AD_in_the_family_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_ad_ip
group2 = [TAU_CAT, AD_IF]

p_ad_family = stat_test(data_type, group1, group2) # Too few samples for the test

### Herpes_zoster_vaccine

In [ ]:
HZV = 'Herpes_zoster_vaccine'

df_high_vs_low_hzv = df_high_vs_low.dropna(subset=[HZV])
grpah_colors = [healthy_color, risk_color]

plot_histplot(df_high_vs_low_hzv, HZV, TAU_CAT, 2, grpah_colors, 'Herpes zoster virus vs Tau 217 category', RISK_FACTOR_DIR + 'Herpes_zoster_virus_vs_Tau_217_category_hist.pdf')

In [ ]:
# Statistical test

data_type = 'Categorical'

group1 = df_high_vs_low_hzv
group2 = [TAU_CAT, HZV]

p_herpes_zoster = stat_test(data_type, group1, group2) # Too few samples for the test

# Statistical Correction
I'm using BH-FDR

In [ ]:
ALPHA = 0.05
METHOD = 'fdr_bh'

In [ ]:
# Statistical Correction
p_values = [p_number_of_risk_factors, p_anxiety, p_bmi, p_heart_disease, p_diabetes, p_hypertenstion, p_hyperlipidemia, p_peridontal, p_uti, p_systemic_inflammatory, p_smoking, p_pack_years, p_alcohol, p_insomnia, p_osa, p_severity_osa, p_physical_activity, p_time_physical_activity, p_cognitive_activity, p_ad_family, p_herpes_zoster]
risk_factors = ['Number of risk factors', 'Anxiety/Depression', 'BMI', 'Heart disease', 'Diabetes', 'Hypertension', 'Hyperlipidemia', 'Periodontal disease', 'Recurrent UTI/pneumonia/other infection', 'Systemic inflammatory disease', 'Smoking', 'Pack years', 'Alcohol consumption', 'Insomnia', 'OSA', 'OSA severity', 'Physical activity', 'Time Physical activity', 'Cognitive activity', 'AD in the family', 'Herpes zoster virus']
p_values = [p for p in p_values if p is not None]
reject, p_values_corrected, _, _ = smm.multipletests(p_values, alpha=ALPHA, method=METHOD)

for i, p in enumerate(p_values):
    if p is not None:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

print("\nSignificant features:")
for i, p in enumerate(p_values):
    if p < ALPHA:
        print(f'{risk_factors[i]}: p-value = {p:.3f} -> adjusted p-value = {p_values_corrected[i]:.3f}')

# Results: 
### Using threshold of 0.53
* significant risk factors:
    * Herpes zoster virus: p-value = 0.002 -> adjusted p-value = 0.023
    * BMI: p-value = 0.006 -> adjusted p-value = 0.040
    * Hyperlipidemia: p-value = 0.002 -> adjusted p-value = 0.023
    No risk factors were signifcant before correction and not after
### Using threshold of 0.449
* significant risk factors:
    * Herpes zoster virus: p-value = 0.000 -> adjusted p-value = 0.010
* significant risk factors before correction and not after:
    * Smoking: p-value = 0.030 -> adjusted p-value = 0.211
    * BMI: p-value = 0.009 -> adjusted p-value = 0.092
